### Data Pre-processing & Transformation

In [1]:
%pip install Sastrawi

import json
import re
from pathlib import Path
from functools import lru_cache

import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.feature_extraction import text as sklearn_text

try:
    from Sastrawi.Stemmer.StemmerFactory import StemmerFactory
    STEMMER_ID = StemmerFactory().create_stemmer()
except Exception:
    STEMMER_ID = None

DATA_PATH = Path("/home/fatih/Documents/Projects/nlp-teori/dataset/ICAR_Text_Extracted.json")
OUT_DIR = Path("/home/fatih/Documents/Projects/nlp-teori/output")
OUT_DIR.mkdir(parents=True, exist_ok=True)
TOP_K_FEATURES = 1000

# Untuk mempercepat: set False dulu. Ubah ke True kalau butuh stemming.
USE_STEMMING = False

Note: you may need to restart the kernel to use updated packages.


In [2]:
# 2) Load dataset JSON
with open(DATA_PATH, "r", encoding="utf-8") as f:
    data = json.load(f)

rows = []
for category, files in data.items():
    if isinstance(files, dict):
        for fname, txt in files.items():
            rows.append({
                "category": category,
                "doc_id": fname,
                "text_raw": str(txt) if txt is not None else ""
            })

df = pd.DataFrame(rows)
print("Jumlah dokumen:", len(df))
df.head()

Jumlah dokumen: 159


,category,doc_id,text_raw
0,Annual Reports,anrep-02003.pdf,\n--- Page 1 ---\nWeb Url-https://icar.gov.in/...
1,Annual Reports,AR-2019-20.pdf,\n--- Page 1 ---\nWeb Url-https://icar.gov.in/...
2,Annual Reports,DARE-Annual-Report-2017-18.pdf,\n--- Page 1 ---\nWeb Url -https://icar.gov.in...
3,Annual Reports,DARE-ICAR-Annual-Report 2016-17.pdf,\n--- Page 1 ---\nWeb Url-https://icar.gov.in/...
4,Annual Reports,DARE-ICAR-AR-2018-19.pdf,\n--- Page 1 ---\nANNUAL REPORT 2018-19 or 04[...


In [3]:
# 3) Data cleaning + normalization + stopword removal
stopwords_en = set(sklearn_text.ENGLISH_STOP_WORDS)
stopwords_id = {
    "dan", "yang", "di", "ke", "dari", "untuk", "pada", "dengan", "atau",
    "ini", "itu", "adalah", "karena", "sebagai", "oleh", "dalam", "juga",
    "agar", "para", "kita", "kami", "mereka", "akan", "telah", "dapat"
}
stopwords_all = stopwords_en.union(stopwords_id)

url_pattern = re.compile(r"http\S+|www\.\S+")
digit_pattern = re.compile(r"\d+")
non_alpha_pattern = re.compile(r"[^a-zA-Z\s]")
space_pattern = re.compile(r"\s+")


def basic_cleaning(text: str) -> str:
    text = text.lower()                                  # lowercase
    text = url_pattern.sub(" ", text)                    # hapus URL
    text = digit_pattern.sub(" ", text)                  # hapus angka
    text = non_alpha_pattern.sub(" ", text)              # hapus tanda baca/simbol
    text = space_pattern.sub(" ", text).strip()          # rapikan spasi
    return text


@lru_cache(maxsize=100_000)
def stem_token(token: str) -> str:
    return STEMMER_ID.stem(token)


def normalize_text(text: str) -> str:
    if not USE_STEMMING or STEMMER_ID is None:
        return text
    tokens = text.split()
    return " ".join(stem_token(t) for t in tokens)


def remove_stopwords(text: str) -> str:
    tokens = [t for t in text.split() if t not in stopwords_all and len(t) > 2]
    return " ".join(tokens)


def preprocess_pipeline(text: str) -> str:
    text = basic_cleaning(text)
    text = normalize_text(text)
    text = remove_stopwords(text)
    return text

In [4]:
# 4) Terapkan preprocessing ke kolom text_clean
df["text_clean"] = df["text_raw"].apply(preprocess_pipeline)

print("Contoh hasil preprocessing:")
df[["category", "doc_id", "text_clean"]].head()

Contoh hasil preprocessing:


,category,doc_id,text_clean
0,Annual Reports,anrep-02003.pdf,page web url dare icar annual report departmen...
1,Annual Reports,AR-2019-20.pdf,page web url page indian council agricultvral ...
2,Annual Reports,DARE-Annual-Report-2017-18.pdf,page web url page indian council agricultural ...
3,Annual Reports,DARE-ICAR-Annual-Report 2016-17.pdf,page web url page indian council agricultural ...
4,Annual Reports,DARE-ICAR-AR-2018-19.pdf,page annual report taaa department agricultura...


In [5]:
# 5) Data transformation: TF-IDF
vectorizer = TfidfVectorizer(
    min_df=2,
    max_df=0.85,
    ngram_range=(1, 2)
)
X = vectorizer.fit_transform(df["text_clean"])
feature_names = np.array(vectorizer.get_feature_names_out())

print("Shape TF-IDF:", X.shape)
print("Jumlah fitur:", len(feature_names))

Shape TF-IDF: (159, 478980)
Jumlah fitur: 478980


In [6]:
# 6) Feature selection: pilih kata relevan (top-k mean TF-IDF)
mean_scores = np.asarray(X.mean(axis=0)).ravel()
top_k = min(TOP_K_FEATURES, len(feature_names))
top_idx = np.argsort(mean_scores)[::-1][:top_k]

X_selected = X[:, top_idx]
selected_features = feature_names[top_idx]

selected_df = pd.DataFrame({
    "feature": selected_features,
    "mean_tfidf": mean_scores[top_idx]
}).sort_values("mean_tfidf", ascending=False)

selected_df.head(20)

,feature,mean_tfidf
0,yield,0.045361
1,farming,0.036702
2,pradesh,0.030826
3,days,0.030401
4,fruit,0.030219
5,rice,0.029290
6,varieties,0.028661
7,cultivation,0.025492
8,variety,0.023897
9,used,0.023844


In [7]:
# 7) Simpan hasil
preprocessed_path = OUT_DIR / "icar_preprocessed.csv"
features_path = OUT_DIR / "icar_selected_features.csv"
tfidf_selected_path = OUT_DIR / "icar_tfidf_selected.csv"

# a) Teks hasil preprocessing
df[["category", "doc_id", "text_clean"]].to_csv(preprocessed_path, index=False)

# b) Fitur terpilih
selected_df.to_csv(features_path, index=False)

# c) Matriks TF-IDF tereduksi
tfidf_df = pd.DataFrame(X_selected.toarray(), columns=selected_features)
tfidf_df.insert(0, "doc_id", df["doc_id"].values)
tfidf_df.to_csv(tfidf_selected_path, index=False)

print("Selesai disimpan:")
print("-", preprocessed_path)
print("-", features_path)
print("-", tfidf_selected_path)

Selesai disimpan:
- /home/fatih/Documents/Projects/nlp-teori/output/icar_preprocessed.csv
- /home/fatih/Documents/Projects/nlp-teori/output/icar_selected_features.csv
- /home/fatih/Documents/Projects/nlp-teori/output/icar_tfidf_selected.csv
